In [ ]:
import os
#os.chdir("/home/abood/datascienceproject")
%pwd

'/home/abood/datascienceproject'

In [6]:
import pandas as pd

data = pd.read_csv("artifacts/data_ingestion/machine.data")
df = pd.DataFrame(data)
df.head()

,adviser,32/60,125,256,6000,256.1,16,128,198,199
0,amdahl,470v/7,29,8000,32000,32,8,32,269,253
1,amdahl,470v/7a,29,8000,32000,32,8,32,220,253
2,amdahl,470v/7b,29,8000,32000,32,8,32,172,253
3,amdahl,470v/7c,29,8000,16000,32,8,16,132,132
4,amdahl,470v/b,26,8000,32000,64,8,32,318,290


In [7]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_path: Path
    alpha: float
    l1_ratio: float
    target_column: str

In [8]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories

In [9]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path=config.train_data_path,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            alpha=params.alpha,
            l1_ratio=params.l1_ratio,
            target_column=schema.name
        )
        return model_trainer_config

In [10]:
import os
import joblib
from src.datascience.utils import logger
from sklearn.linear_model import ElasticNet
import pandas as pd

In [11]:
class ModelTrainer:
    def __init__(self , config: ModelTrainerConfig):
        self.config = config
    def train(self):
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)

        train_x = train_data.drop(self.config.target_column, axis=1)
        test_x = test_data.drop(self.config.target_column, axis=1)
        train_y = train_data[self.config.target_column]
        test_y = test_data[self.config.target_column]

        lr = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio , random_state=42)
        lr.fit(train_x, train_y)
        joblib.dump(lr, os.path.join(self.config.root_dir , self.config.model_path))

In [12]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()
except Exception as e:
    logger.exception(e)

[2026-05-06 13:24:03,374: INFO: common]: yaml file: config/config.yaml loaded successfully
[2026-05-06 13:24:03,383: INFO: common]: yaml file: params.yaml loaded successfully
[2026-05-06 13:24:03,391: INFO: common]: yaml file: schema.yaml loaded successfully
[2026-05-06 13:24:03,394: INFO: common]: created directory at: artifacts
[2026-05-06 13:24:03,395: INFO: common]: created directory at: artifacts/model_trainer
